# CrimeScope ML — 06. Export for Backend

Export enriched scored data, tract boundaries, ACS, and pipeline stats as JSON
to UC Volume for backend consumption.

In [ ]:
import json
from pyspark.sql import functions as F

spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

VOLUME_PATH = "/Volumes/varanasi/default/ml_data"

# 1. Export tract_risk_scores (enriched with sub-scores)
scores_pdf = spark.table("varanasi.default.tract_risk_scores").toPandas()
scores_json = scores_pdf.to_json(orient="records", date_format="iso")
with open(f"{VOLUME_PATH}/tract_risk_scores.json", "w") as f:
    f.write(scores_json)
print(f"Exported {len(scores_pdf)} tract scores ({len(scores_pdf.columns)} columns)")
print(f"  Columns: {list(scores_pdf.columns)}")

# 2. Export tract boundaries
tracts_pdf = spark.table("varanasi.default.cook_tract_boundaries").select(
    "tract_geoid", "NAMELSAD", "wkt", "ALAND"
).toPandas()
tracts_json = tracts_pdf.to_json(orient="records")
with open(f"{VOLUME_PATH}/cook_tract_boundaries.json", "w") as f:
    f.write(tracts_json)
print(f"Exported {len(tracts_pdf)} tract boundaries")

# 3. Export ACS data
acs_pdf = spark.table("varanasi.default.tract_acs_population").toPandas()
acs_json = acs_pdf.to_json(orient="records")
with open(f"{VOLUME_PATH}/tract_acs_population.json", "w") as f:
    f.write(acs_json)
print(f"Exported {len(acs_pdf)} ACS records")

# 4. Export feature table stats for trust passport
feat = spark.table("varanasi.default.tract_crime_features")
stats = feat.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("tract_geoid").alias("n_tracts"),
    F.countDistinct("month_start").alias("n_months"),
    F.min("month_start").alias("data_start"),
    F.max("month_start").alias("data_end"),
).toPandas()

# Add model info to stats
risk_scores = spark.table("varanasi.default.tract_risk_scores")
model_stats = risk_scores.select(
    F.count("*").alias("scored_tracts"),
    F.round(F.avg("risk_score"), 1).alias("avg_risk_score"),
    F.round(F.avg("violent_score"), 1).alias("avg_violent_score"),
    F.round(F.avg("property_score"), 1).alias("avg_property_score"),
    F.first("model_version").alias("model_version"),
    F.first("scored_at").alias("last_scored_at"),
).toPandas()

combined_stats = pd.concat([stats, model_stats], axis=1)
stats_json = combined_stats.to_json(orient="records", date_format="iso")
with open(f"{VOLUME_PATH}/pipeline_stats.json", "w") as f:
    f.write(stats_json)
print(f"Exported pipeline stats")

# 5. Verify the enriched output
import pandas as pd
sample = scores_pdf.sort_values("risk_score", ascending=False).head(3)
for _, row in sample.iterrows():
    print(f"\nTract {row['tract_geoid']}: score={row['risk_score']}, " +
          f"violent={row['violent_score']}, property={row['property_score']}, " +
          f"tier={row['risk_tier']}, trend={row['trend_direction']}")
    drivers = json.loads(row['top_drivers_json'])
    for d in drivers[:3]:
        print(f"  {d['direction']:>4} {d['label']}: SHAP={d['shap_value']:.4f}")

print(f"\nAll exports complete. Files in UC Volume:")
print(f"  {VOLUME_PATH}/tract_risk_scores.json")
print(f"  {VOLUME_PATH}/cook_tract_boundaries.json")
print(f"  {VOLUME_PATH}/tract_acs_population.json")
print(f"  {VOLUME_PATH}/pipeline_stats.json")